# 16 认识图像分类数据集

本节目标：认识 Fashion-MNIST、训练集、测试集、batch、类别标签和图像张量。

## 1. 最小概念

- Fashion-MNIST：10 类服饰灰度图像数据集，每张图是 `28 x 28`。
- 训练集：给模型学习的数据。
- 测试集：训练后用来评估泛化能力的数据。
- batch：一次送进模型的一小批图片。
- 类别标签：每张图片对应的类别编号，范围是 `0` 到 `9`。
- 图像张量：在 PyTorch 中常见 shape 是 `(batch, channel, height, width)`。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset

try:
    from torchvision import datasets, transforms
    HAS_TORCHVISION = True
except Exception as exc:
    HAS_TORCHVISION = False
    TORCHVISION_ERROR = exc


def load_fashion_mnist(train_limit=1024, test_limit=512, batch_size=256):
    data_root = Path("../data")
    if HAS_TORCHVISION:
        try:
            transform = transforms.ToTensor()
            train_full = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
            test_full = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)
            train_data = Subset(train_full, range(min(train_limit, len(train_full))))
            test_data = Subset(test_full, range(min(test_limit, len(test_full))))
            source = "Fashion-MNIST"
        except Exception as exc:
            print("Fashion-MNIST 下载或读取失败，改用同 shape 的随机数据继续练习:", repr(exc))
            train_images = torch.rand(train_limit, 1, 28, 28)
            train_labels = torch.randint(0, 10, (train_limit,))
            test_images = torch.rand(test_limit, 1, 28, 28)
            test_labels = torch.randint(0, 10, (test_limit,))
            train_data = TensorDataset(train_images, train_labels)
            test_data = TensorDataset(test_images, test_labels)
            source = "synthetic fallback"
    else:
        print("torchvision 不可用，改用同 shape 的随机数据继续练习:", repr(TORCHVISION_ERROR))
        train_images = torch.rand(train_limit, 1, 28, 28)
        train_labels = torch.randint(0, 10, (train_limit,))
        test_images = torch.rand(test_limit, 1, 28, 28)
        test_labels = torch.randint(0, 10, (test_limit,))
        train_data = TensorDataset(train_images, train_labels)
        test_data = TensorDataset(test_images, test_labels)
        source = "synthetic fallback"

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, source

class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
train_loader, test_loader, source = load_fashion_mnist(train_limit=512, test_limit=256, batch_size=64)
images, labels = next(iter(train_loader))

print("数据来源:", source)
print("一个 batch 的图像 shape:", tuple(images.shape))
print("一个 batch 的标签 shape:", tuple(labels.shape))
print("图像 dtype:", images.dtype)
print("标签 dtype:", labels.dtype)
print("前 10 个标签:", labels[:10])

In [ ]:
assets_dir = Path("assets")
assets_dir.mkdir(exist_ok=True)
output_file = assets_dir / "16-fashion-mnist-samples.png"

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, img, label in zip(axes.flatten(), images[:8], labels[:8]):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[int(label)] if int(label) < len(class_names) else str(int(label)))
    ax.axis("off")
plt.tight_layout()
plt.savefig(output_file, dpi=160)
plt.show()
print("样本图已保存:", output_file)

## 小结

图像分类里，模型输入通常是四维张量：`(batch, channel, height, width)`。Fashion-MNIST 的单张图是灰度图，所以 channel 是 `1`。